# 07 - Imitation Learning

## What / Why / How

**What we are trying to do:** Train a policy from demonstrations using behavior cloning and action chunks.

**Why this matters:** Most practical robot learning starts from examples because random real-world exploration is slow and unsafe.

**How we will do it:** Generate expert trajectories, fit a supervised policy, roll it out closed-loop, and compare single actions with short action chunks.

## Goal

Learn the most practical first training method in robotics: imitation learning.

You will:

- Generate expert demonstrations.
- Train a behavior cloning policy.
- See why closed-loop evaluation matters.
- Build a tiny action-chunking policy.

This connects to ACT, ALOHA, Mobile ALOHA, Diffusion Policy, OpenVLA, SmolVLA, and pi0-style robot policies.

In [ ]:
import math
import random
from collections import defaultdict

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see the plot: pip install -r requirements.txt")

## Generate Demonstrations

In [ ]:
rng = np.random.default_rng(0)
target = np.array([1.0, 1.0])

def expert_action(pos):
    return np.clip(0.7 * (target - pos), -0.35, 0.35)

demos = []
for _ in range(60):
    pos = rng.uniform(-1.2, 1.2, size=2)
    states, actions = [], []
    for _ in range(35):
        action = expert_action(pos)
        states.append(pos.copy())
        actions.append(action.copy())
        pos = pos + action + rng.normal(0, 0.01, size=2)
    demos.append({"states": np.array(states), "actions": np.array(actions)})

print("num demonstrations:", len(demos))
print("one state/action:", demos[0]["states"][0], demos[0]["actions"][0])

## Behavior Cloning With Linear Regression

In [ ]:
def features(pos):
    x, y = pos
    return np.array([x, y, target[0], target[1], 1.0])

X = []
Y = []
for demo in demos:
    for s, a in zip(demo["states"], demo["actions"]):
        X.append(features(s))
        Y.append(a)
X = np.array(X)
Y = np.array(Y)

W, *_ = np.linalg.lstsq(X, Y, rcond=None)

def cloned_policy(pos):
    return np.clip(features(pos) @ W, -0.35, 0.35)

print("weights shape:", W.shape)
for test in [np.array([-1, -1]), np.array([0.2, 1.4]), np.array([1.2, -0.8])]:
    print("pos", test, "expert", expert_action(test), "clone", cloned_policy(test))

## Closed-Loop Evaluation

In [ ]:
def rollout(policy, start, steps=35):
    pos = np.array(start, dtype=float)
    route = [pos.copy()]
    for _ in range(steps):
        action = policy(pos)
        pos = pos + action
        route.append(pos.copy())
    return np.array(route)

starts = [np.array([-1.0, -0.8]), np.array([1.4, -1.0]), np.array([-1.2, 1.3])]
for start in starts:
    route = rollout(cloned_policy, start)
    print("start", start, "final error", round(float(np.linalg.norm(route[-1] - target)), 3))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 5))
    for start in starts:
        route = rollout(cloned_policy, start)
        plt.plot(route[:, 0], route[:, 1], marker="o", markersize=2)
    plt.scatter([target[0]], [target[1]], c="tab:red", s=80, label="target")
    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.title("Behavior cloning rollouts")
    plt.show()
else:
    plot_unavailable()

## Action Chunking

In [ ]:
horizon = 4
Xc, Yc = [], []
for demo in demos:
    states = demo["states"]
    actions = demo["actions"]
    for i in range(len(states) - horizon):
        Xc.append(features(states[i]))
        Yc.append(actions[i:i+horizon].reshape(-1))
Xc = np.array(Xc)
Yc = np.array(Yc)
W_chunk, *_ = np.linalg.lstsq(Xc, Yc, rcond=None)

def chunk_policy(pos):
    chunk = (features(pos) @ W_chunk).reshape(horizon, 2)
    return np.clip(chunk[0], -0.35, 0.35)

route = rollout(chunk_policy, [-1.1, -0.9])
print("chunk policy final error:", round(float(np.linalg.norm(route[-1] - target)), 3))
print("first predicted action chunk:")
print((features(np.array([-1.1, -0.9])) @ W_chunk).reshape(horizon, 2))

## What To Notice

Behavior cloning can work very well when demonstrations cover the states the robot will visit. It fails when the robot drifts into states not present in the data.

Modern systems reduce this problem using:

- More diverse demos.
- Better sensors and proprioception.
- Action chunks.
- Diffusion or flow policies that model multiple valid futures.
- Human correction loops such as DAgger-style data collection.

## Exercises

1. Reduce the number of demos to 5. What fails?
2. Add noisy actions to the demonstrations.
3. Change `target` and retrain.
4. Make a validation set and report mean final error.